In [ ]:
from typing import TypedDict, Annotated

from IPython.core.magics import config
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, BaseMessage, AIMessage
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END, state
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
llm = ChatGroq(model="qwen/qwen3-32b", temperature=0.4, reasoning_format="hidden")

In [ ]:
def chat_node(state: ChatState) -> ChatState:
    print(list(state["messages"]))
    response_llm = llm.invoke(input=state["messages"])
    return {"messages": [AIMessage(content=response_llm.content)]}

In [ ]:
checkpointer = InMemorySaver()
graph = StateGraph(ChatState)


graph.add_node("chat_node", chat_node)

graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)

workflow = graph.compile(checkpointer= checkpointer)



In [ ]:
thread_id = "mychat-ui"
config = {"configurable": {"thread_id": thread_id}}

while True:
    user_message = input("Human: ")
    initial_state = {"messages": [HumanMessage(content=user_message)]}

    print(f"Human: {user_message}")
    if user_message.lower().strip() in ["bye", "exit", "quit"]:
        break

    response = workflow.invoke(input= initial_state, config= config)
    print(f"AI: {response["messages"][-1].content} \n")

In [ ]:
print("=== State History ===")
for snapshot in workflow.get_state_history(config):
    print(f"--- Step {snapshot.metadata['step']} | {snapshot.created_at} ---")
    print("Next:  ", snapshot.next)
    print("Values:", {k: v[:60] + "..." if isinstance(v, str) and len(v) > 60 else v
                     for k, v in snapshot.values.items()})
    print()